In [0]:
%sql
USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:

from pyspark.sql import functions as F

catalogo = "medalhao"
silver_db = "silver"
gold_db = "gold"

In [0]:
# Carregamento de todas as tabelas limpas da camada Silver
df_info = spark.table(f"{catalogo}.{silver_db}.tb_info_filmes")
df_fin = spark.table(f"{catalogo}.{silver_db}.tb_financeiro_filmes")
df_metrics = spark.table(f"{catalogo}.{silver_db}.tb_metricas_engajamento")
df_reviews = spark.table(f"{catalogo}.{silver_db}.tb_avaliacoes_usuarios")
df_generos = spark.table(f"{catalogo}.{silver_db}.tb_generos")
df_pessoas_empresas = spark.table(f"{catalogo}.{silver_db}.tb_pessoas_empresas")

print("Todas as tabelas da Silver foram carregadas com SUCESSO para o ambiente da Gold!")

Todas as tabelas da Silver foram carregadas com SUCESSO para o ambiente da Gold!


### Criação da Dimensão de Filmes (dim_movies)
Constrói o catálogo central de metadados dos filmes, gerando a Chave Substituta (sk_movie_id) sequencial por ordem natural de ID e persistindo em formato Delta.

In [0]:
from pyspark.sql.window import Window

# Geração da Chave Substituta para a Dimensão de Filmes
window_dim_movie = Window.orderBy("id_filme")

df_dim_movies = (
    df_info
    .select(
        F.col("id_filme"),
        F.col("titulo"),
        F.col("data_lancamento"),
        F.col("ano_lancamento"),
        F.col("duracao_minutos"),
        F.col("idioma_original"),
        F.col("status_filme"),
        F.col("sinopse")
    )
    .dropDuplicates(["id_filme"])
    .withColumn("sk_movie_id", F.row_number().over(window_dim_movie).cast("bigint"))
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

# Persistência da Dimensão de Filmes em Delta na Camada Gold
(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db}.dim_movies")
)

print("Tabela gold.dim_movies criada e persistida com SUCESSO!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela gold.dim_movies criada e persistida com SUCESSO!


### Criação das Dimensões Periféricas
Extrai catálogos únicos e deduplicados para Gêneros, Pessoas (filtrando estritamente por Ator, Diretor e Roteirista) e Produtoras, gerando suas respectivas Surrogate Keys.

In [0]:
from pyspark.sql.window import Window

# 1. Dimensão de Gêneros (gold.dim_genres)
window_genre = Window.orderBy("nome_genero")

df_dim_genres = (
    df_generos
    .select(F.trim(F.col("genero")).alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .dropDuplicates(["nome_genero"])
    .withColumn("sk_genre_id", F.row_number().over(window_genre).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_genres")
print("Tabela gold.dim_genres criada e persistida com SUCESSO!")

# 2. Dimensão de Pessoas (gold.dim_people)
window_person = Window.orderBy("nome_pessoa", "tipo_pessoa")

df_dim_people = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
    .withColumn("sk_person_id", F.row_number().over(window_person).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_people")
print("Tabela gold.dim_people criada e persistida com SUCESSO!")

# 3. Dimensão de Produtoras (gold.dim_companies)
window_company = Window.orderBy("nome_produtora")

df_dim_companies = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.trim(F.col("nome_entidade")).alias("nome_produtora"))
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", F.row_number().over(window_company).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_companies")
print("Tabela gold.dim_companies criada e persistida com SUCESSO!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela gold.dim_genres criada e persistida com SUCESSO!
Tabela gold.dim_people criada e persistida com SUCESSO!
Tabela gold.dim_companies criada e persistida com SUCESSO!


### Criação da Dimensão de Avaliações (dim_reviews)
Agrega as avaliações de usuários por filme, calculando a contagem total e a nota média arredondada, mapeando a chave estrangeira para o respectivo filme.

In [0]:
from pyspark.sql.window import Window

df_dim_movies_gold = spark.sql(f"SELECT sk_movie_id, id_filme FROM {catalogo}.{gold_db}.dim_movies")

# Agregação das avaliações de usuários por filme (1 registro por filme avaliado)
df_reviews_agregado = (
    df_reviews
    .groupBy("id_filme")
    .agg(
        F.count("nota_usuario").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )
)

window_review = Window.orderBy("id_filme")

df_dim_reviews = (
    df_reviews_agregado
    .join(df_dim_movies_gold, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(window_review).cast("bigint"))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

# Persistência da Dimensão de Reviews em Delta na Gold
(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db}.dim_reviews")
)

print("Tabela gold.dim_reviews criada e persistida com SUCESSO!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela gold.dim_reviews criada e persistida com SUCESSO!


### Criação das Tabelas-Ponte (Bridge Tables)
Estabelece os relacionamentos N:N entre os filmes e suas dimensões multivaloradas (Gêneros, Pessoas e Produtoras) preservando o grão unívoco da Fato.

In [0]:
df_movies_sk = spark.sql(f"SELECT sk_movie_id, id_filme FROM {catalogo}.{gold_db}.dim_movies")
df_genres_sk = spark.sql(f"SELECT sk_genre_id, nome_genero FROM {catalogo}.{gold_db}.dim_genres")
df_people_sk = spark.sql(f"SELECT sk_person_id, nome_pessoa, tipo_pessoa FROM {catalogo}.{gold_db}.dim_people")
df_companies_sk = spark.sql(f"SELECT sk_company_id, nome_produtora FROM {catalogo}.{gold_db}.dim_companies")

# Bridge Filme x Gênero (gold.bridge_movie_genre)
df_bridge_genre = (
    df_generos
    .select(
        F.col("id_filme"),
        F.trim(F.col("genero")).alias("nome_genero")
    )
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_genres_sk, on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates(["sk_movie_id", "sk_genre_id"])
)

df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.bridge_movie_genre")
print("Tabela gold.bridge_movie_genre criada e persistida com SUCESSO!")

# Bridge Filme x Pessoa / Papel (gold.bridge_movie_person)
df_bridge_person = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("id_filme"),
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_people_sk, on=["nome_pessoa", "tipo_pessoa"], how="inner")
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates(["sk_movie_id", "sk_person_id"])
)

df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.bridge_movie_person")
print("Tabela gold.bridge_movie_person criada e persistida com SUCESSO!")

# Bridge Filme x Produtora (gold.bridge_movie_company)
df_bridge_company = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("id_filme"),
        F.trim(F.col("nome_entidade")).alias("nome_produtora")
    )
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_companies_sk, on="nome_produtora", how="inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates(["sk_movie_id", "sk_company_id"])
)

df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.bridge_movie_company")
print("Tabela gold.bridge_movie_company criada e persistida com SUCESSO!")

Tabela gold.bridge_movie_genre criada e persistida com SUCESSO!
Tabela gold.bridge_movie_person criada e persistida com SUCESSO!
Tabela gold.bridge_movie_company criada e persistida com SUCESSO!


### Criação da Tabela Fato (fact_movies_performance)
Centraliza as métricas financeiras (orçamento, receita e lucro em USD e BRL) e indicadores de engajamento por filme, associando à dimensão principal via Surrogate Key.

In [0]:
df_dim_movies_gold = spark.sql(f"SELECT sk_movie_id, id_filme FROM {catalogo}.{gold_db}.dim_movies")

# Cruzamento e Derivação de Métricas de Performance na Fato
df_fact_base = (
    df_dim_movies_gold
    .join(df_fin, on="id_filme", how="left")
    .join(df_metrics, on="id_filme", how="left")
)

# Projeção estruturada e cálculo de Lucro (USD e BRL)
df_fact_movies_performance = df_fact_base.select(
    F.col("sk_movie_id"),
    
    # Métricas Financeiras (USD)
    F.col("orcamento_usd"),
    F.col("receita_usd"),
    F.when(
        F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull(),
        (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)")
    ).otherwise(F.lit(None)).alias("lucro_usd"),
    
    # Métricas Financeiras (BRL)
    F.col("orcamento_brl"),
    F.col("receita_brl"),
    F.when(
        F.col("receita_brl").isNotNull() & F.col("orcamento_brl").isNotNull(),
        (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)")
    ).otherwise(F.lit(None)).alias("lucro_brl"),
    
    # Métricas de Engajamento
    F.col("popularidade"),
    F.col("nota_media_tmdb"),
    F.col("qtd_votos_tmdb"),
    F.col("nota_media_imdb"),
    F.col("qtd_votos_imdb")
)

# Persistência da Tabela Fato em Delta na Camada Gold
(
    df_fact_movies_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db}.fact_movies_performance")
)

print("Tabela gold.fact_movies_performance criada e persistida com SUCESSO!")

Tabela gold.fact_movies_performance criada e persistida com SUCESSO!


### Geração da Tabela de Contexto GenAI
Monta o documento aplicando tratamentos defensivos contra nulos (coalesce) e conversões de tipo

In [0]:
df_dim_movies = spark.sql(f"SELECT sk_movie_id, id_filme, titulo, ano_lancamento, sinopse FROM {catalogo}.{gold_db}.dim_movies")
df_fact_perf = spark.sql(f"SELECT sk_movie_id, orcamento_usd, receita_usd FROM {catalogo}.{gold_db}.fact_movies_performance")
df_bridge_person = spark.sql(f"SELECT sk_movie_id, sk_person_id FROM {catalogo}.{gold_db}.bridge_movie_person")
df_dim_people = spark.sql(f"SELECT sk_person_id, nome_pessoa, tipo_pessoa FROM {catalogo}.{gold_db}.dim_people")

# Extrair Diretores por filme
df_diretores = (
    df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("diretor_nome"))
)

# Extrair Atores principais por filme
df_atores = (
    df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("atores_nomes"))
)

# Consolidar as informações com estratégias de Fallback para evitar nulos
df_context_base = (
    df_dim_movies
    .join(df_fact_perf, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
    .join(df_atores, on="sk_movie_id", how="left")
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo"),
        F.coalesce(F.col("titulo"), F.lit("Título desconhecido")).alias("t"),
        F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano desconhecido")).alias("a"),
        F.coalesce(F.format_string("US$ %,.2f", F.col("receita_usd").cast("double")), F.lit("receita não informada")).alias("r"),
        F.coalesce(F.format_string("US$ %,.2f", F.col("orcamento_usd").cast("double")), F.lit("orçamento não informado")).alias("o"),
        F.coalesce(F.col("atores_nomes"), F.lit("elenco não informado")).alias("at"),
        F.coalesce(F.col("diretor_nome"), F.lit("direção não informada")).alias("d"),
        F.coalesce(F.col("sinopse"), F.lit("Sinopse não informada")).alias("s")
    )
)

# Construção do documento em frase corrida seguindo o template
df_genai_final = df_context_base.select(
    F.col("movie_id"),
    F.col("titulo").alias("title"),
    F.concat_ws(
        "",
        F.lit("O filme "), F.col("t"),
        F.lit(", lançado no ano de "), F.col("a"),
        F.lit(", faturou "), F.col("r"),
        F.lit(" e teve um custo de "), F.col("o"),
        F.lit(". Estrelado por "), F.col("at"),
        F.lit(" e dirigido por "), F.col("d"),
        F.lit(", o filme possui a seguinte sinopse: "), F.col("s"),
        F.lit(".")
    ).alias("llm_context_document")
)

# Persistência da tabela de contexto em Delta na Gold
(
    df_genai_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db}.gold_genai_movies_context")
)

print("Tabela gold.gold_genai_movies_context criada e persistida com SUCESSO!")

Tabela gold.gold_genai_movies_context criada e persistida com SUCESSO!


### Desafio de Analytics
Consultas analíticas para responder às 6 perguntas de negócio utilizando as tabelas da camada Gold.

In [0]:
# Desafio de Analytics: Consultas de Negócio


# 1. Receita total (em R$) somada de todos os filmes da base
display(spark.sql(f"""
    SELECT sum(receita_brl) AS receita_total_brl 
    FROM {catalogo}.{gold_db}.fact_movies_performance
"""))

# 2. Top 5 filmes com maior popularidade (título e popularidade)
display(spark.sql(f"""
    SELECT 
        m.titulo, 
        f.popularidade 
    FROM {catalogo}.{gold_db}.fact_movies_performance f
    JOIN {catalogo}.{gold_db}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

# 3. Contagem de filmes por gênero (ordenado do maior para o menor)
display(spark.sql(f"""
    SELECT 
        g.nome_genero, 
        count(b.sk_movie_id) AS qtd_filmes
    FROM {catalogo}.{gold_db}.bridge_movie_genre b
    JOIN {catalogo}.{gold_db}.dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

# 4. Top 10 filmes de maior receita (título, receita USD, BRL e RANK)
display(spark.sql(f"""
    WITH ranked_movies AS (
        SELECT 
            m.titulo,
            f.receita_usd,
            f.receita_brl,
            RANK() OVER (ORDER BY f.receita_usd DESC NULLS LAST) AS ranking
        FROM {catalogo}.{gold_db}.fact_movies_performance f
        JOIN {catalogo}.{gold_db}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT * FROM ranked_movies
    WHERE ranking <= 10
    ORDER BY ranking
"""))

# 5. Ator com maior quantidade de participações nos últimos 2 anos
display(spark.sql(f"""
    WITH max_date AS (
        SELECT max(data_lancamento) AS dt_max 
        FROM {catalogo}.{gold_db}.dim_movies 
        WHERE data_lancamento <= current_date()
    )
    SELECT 
        p.nome_pessoa AS ator,
        count(b.sk_movie_id) AS qtd_participacoes
    FROM {catalogo}.{gold_db}.bridge_movie_person b
    JOIN {catalogo}.{gold_db}.dim_people p ON b.sk_person_id = p.sk_person_id
    JOIN {catalogo}.{gold_db}.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN max_date md
    WHERE p.tipo_pessoa = 'Ator'
      AND m.data_lancamento >= add_months(md.dt_max, -24)
      AND m.data_lancamento <= md.dt_max
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC
    LIMIT 1
"""))

# 6. Produtora com maior Lucro em USD e BRL nos últimos 5 anos
display(spark.sql(f"""
    WITH max_date AS (
        SELECT max(data_lancamento) AS dt_max 
        FROM {catalogo}.{gold_db}.dim_movies 
        WHERE data_lancamento <= current_date()
    )
    SELECT 
        c.nome_produtora,
        sum(f.lucro_usd) AS lucro_total_usd,
        sum(f.lucro_brl) AS lucro_total_brl
    FROM {catalogo}.{gold_db}.bridge_movie_company b
    JOIN {catalogo}.{gold_db}.dim_companies c ON b.sk_company_id = c.sk_company_id
    JOIN {catalogo}.{gold_db}.fact_movies_performance f ON b.sk_movie_id = f.sk_movie_id
    JOIN {catalogo}.{gold_db}.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN max_date md
    WHERE m.data_lancamento >= add_months(md.dt_max, -60)
      AND m.data_lancamento <= md.dt_max
    GROUP BY c.nome_produtora
    ORDER BY sum(f.lucro_usd) DESC NULLS LAST
    LIMIT 1
"""))

receita_total_brl
607471030662.95


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


nome_genero,qtd_filmes
Drama,30353
Documentary,18560
Comedy,17312
Thriller,9406
Horror,9128
Romance,6992
Action,5547
Crime,4312
Animation,4154
Tv Movie,3674


titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,11094720000.00,1
Avatar: The Way of Water,2320250281.00,12390136500.54,2
AVENGERS: INFINITY WAR,2052415039.00,7190430847.63,3
spider-man: no way home,1921847111.00,10977782882.74,4
The Lion King,1663075401.00,6227552146.58,5
Top Gun: Maverick,1488732821.00,7160804869.01,6
Barbie,1428545028.00,6856159007.38,7
The Super Mario Bros. Movie,1355725263.00,6838413799.10,8
Black Panther,1349926083.00,4429782441.36,9
Star Wars: The Last Jedi,1332698830.00,4401904235.49,10


ator,qtd_participacoes
Kevin Hart,66


nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5609033667.00,28587272726.64
